# Phase B1 — run NTSA by label rule

Define `label_rule`, then call `ntsa_run(rule)` for each rule. Each invocation loads and validates **only that rule**, resumes matching checkpoints, runs the existing cores, and saves results under `results/ntsa_label_sensitive/<rule>/`.

Only cells containing explicit `ntsa_run(...)` calls launch computation. Configuration and function-definition cells do not run NTSA. No preprocessing, relabeling, resegmentation, P0 computation or Awake–Drowsy comparison is performed.

`S6` is accepted as a folder name but needs its own Phase B0 data and entries in the B0 summaries. The currently available datasets are T30/T60/S3/S5; S6 is not silently replaced by S5.

In [1]:
from pathlib import Path
import ast
import csv
import hashlib
import inspect
import json
import os
import re
import sys
import time
import warnings
import nbformat
import numpy as np
import pandas as pd
import scipy
from IPython.display import display
from joblib import Parallel, delayed, parallel_config

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "phase1/src").is_dir())
PHASE1_DIR = PROJECT_ROOT / "phase1"
INPUT_DIR = PHASE1_DIR / "results/data_label_sensitive"
OUTPUT_DIR = PHASE1_DIR / "results/ntsa_label_sensitive"
sys.path.insert(0, str(PHASE1_DIR / "src"))
from prediction.simplex_projection import simplex_projection, prediction_metrics
from rqa.rqa import run_rqa
from chaos.lyapunov import compute_rosenstein_lle

M = 8
TAU_S = 0.16
SIMPLEX_THEILER_S = 1.0
TARGET_RR = 0.02
L_MIN = V_MIN = 2
FIT_START_S, FIT_END_S = 0.80, 1.30
MAX_FOLLOW_S = 5.0
MIN_INITIAL_PAIRS, MIN_FIT_PAIRS = 50, 30
MIN_R2 = 0.90
N_JOBS = min(6, os.cpu_count() or 1)

# Read only the literal frozen horizon vector, without executing the primary notebook.
PRIMARY_SIMPLEX = PHASE1_DIR / "notebook/simplex_projection.ipynb"
horizon_vectors = []
for cell in nbformat.read(PRIMARY_SIMPLEX, as_version=4).cells:
    if cell.cell_type != "code":
        continue
    for statement in ast.parse(cell.source).body:
        if isinstance(statement, ast.Assign) and any(isinstance(t, ast.Name) and t.id == "HORIZON_SECONDS" for t in statement.targets):
            horizon_vectors.append(ast.literal_eval(statement.value.args[0]))
assert len(horizon_vectors) == 1, "Expected exactly one primary horizon definition"
HORIZON_SECONDS = np.array(horizon_vectors[0], dtype=float)
assert len(HORIZON_SECONDS) == 18 and HORIZON_SECONDS[0] == .04 and HORIZON_SECONDS[-1] == 4.
assert np.all(np.diff(HORIZON_SECONDS) > 0)
print("Frozen primary horizons (s):", HORIZON_SECONDS.tolist())
print(f"Workers: {N_JOBS}; output: {OUTPUT_DIR}")

Frozen primary horizons (s): [0.04, 0.08, 0.12, 0.16, 0.2, 0.28, 0.4, 0.6, 0.8, 1.0, 1.2, 1.6, 2.0, 2.4, 2.8, 3.2, 3.6, 4.0]
Workers: 6; output: /home/vutu0809/Desktop/NTSA_Foundation/phase1/results/ntsa_label_sensitive


## Fixed computation and execution status

- Embedding: `m=8`, `tau_samples=round(0.16 × fs)` from each saved window's frequency.
- Simplex: 18 exact primary horizons, Euclidean distance, `m+1=9` neighbors, exponential weights, leave-one-out, Theiler `round(1 × fs)`. As in primary, only Simplex receives a temporary z-score copy (population standard deviation); NRMSE uses scale 1. `Mean_CC`/`Mean_NRMSE` are arithmetic means across all 18 horizons. Incomplete/nonfinite horizons cause `simplex_success=False` and no partial mean.
- RQA: original processed samples, Euclidean distance, target RR 0.02, Theiler `(m−1) × tau_samples`, `lmin=vmin=2`. Lmean and TT retain the core's units of samples/line points. Achieved RR and tie diagnostics are preserved.
- LLE: original processed samples, spectral mean-period Theiler estimated independently per window, fit 0.80–1.30 s, follow time 5 s, 50 initial/30 fit pairs as in primary. LLE units are s⁻¹. Requested and actually sampled fit endpoints are recorded separately.
- `lle_success` means finite LLE and fit R² were computed, independently of QC. Both R² flags and the core's QC reason remain in the output; neither low R² nor a negative LLE removes a row. No threshold at 0.95 replaces primary QC at 0.90.
- Per-method exceptions/nonfinite outcomes are recorded separately; other methods still run. Input extraction failures retain the window metadata and report all three methods as unsuccessful.

In [2]:
def digest(path):
    with Path(path).open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()

def read_table(path):
    return pd.read_csv(path, dtype={"session":str, "subject":str}, float_precision="round_trip")

def as_bool(series):
    values = series.astype(str).str.lower()
    assert values.isin(["true", "false"]).all(), "Invalid boolean flag"
    return values.eq("true")

METRICS = ["Mean_CC", "Mean_NRMSE", "DET", "Lmean", "LAM", "TT", "LLE", "LLE_fit_R2"]
RESULT_FIELDS = METRICS + ["LLE_fit_start_s", "LLE_fit_end_s", "LLE_pass_R2_090", "LLE_pass_R2_095",
    "simplex_success", "rqa_success", "lle_success", "simplex_error", "rqa_error", "lle_error", "error_message",
    "simplex_horizons_completed", "simplex_horizon_metrics", "simplex_theiler_samples", "rqa_theiler_samples",
    "rqa_achieved_rr", "rqa_rr_exact", "rqa_zero_distance_fraction", "rqa_epsilon",
    "LLE_theiler_samples", "LLE_theiler_s", "LLE_n_pairs_initial", "LLE_n_pairs_fit_min", "LLE_core_valid", "LLE_qc_reason",
    "m", "tau_s", "tau_samples", "requested_fit_start_s", "requested_fit_end_s", "max_follow_s", "target_rr",
    "elapsed_s", "warnings", "signal_extraction_success", "signal_sha256", "computation_key", "computed_for_window_id",
    "reused_identical_signal", "configuration_hash"]

def compute_metrics(signal, fs):
    """Direct public-core calls; return scalar metadata only."""
    started = time.perf_counter()
    tau = int(round(TAU_S * fs))
    out = {k: np.nan for k in RESULT_FIELDS}
    out.update(m=M, tau_s=TAU_S, tau_samples=tau, requested_fit_start_s=FIT_START_S,
        requested_fit_end_s=FIT_END_S, max_follow_s=MAX_FOLLOW_S, target_rr=TARGET_RR,
        simplex_success=False, rqa_success=False, lle_success=False, LLE_pass_R2_090=False, LLE_pass_R2_095=False,
        LLE_core_valid=False, simplex_error="", rqa_error="", lle_error="", error_message="", LLE_qc_reason="",
        simplex_horizons_completed=0, simplex_horizon_metrics="[]", signal_extraction_success=True,
        simplex_theiler_samples=round(SIMPLEX_THEILER_S*fs), rqa_theiler_samples=(M-1)*tau)
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        try:
            scale = float(np.std(signal))
            if not np.isfinite(scale) or scale <= 0:
                raise ValueError("Simplex requires nonzero window variance")
            z = (signal - np.mean(signal)) / scale
            horizon_metrics = []
            for seconds in HORIZON_SECONDS:
                horizon = int(round(seconds*fs))
                result = simplex_projection(z, tau=tau, m=M, horizon=horizon, theiler_window=round(SIMPLEX_THEILER_S*fs))
                metrics = prediction_metrics(result, nrmse_scale=1.0)
                horizon_metrics.append(dict(horizon_s=float(seconds), horizon_samples=horizon,
                    cc=float(metrics.cc) if np.isfinite(metrics.cc) else None,
                    nrmse=float(metrics.nrmse) if np.isfinite(metrics.nrmse) else None,
                    n_valid=metrics.n_valid, n_targets=len(result.y_true)))
                out["simplex_horizons_completed"] = len(horizon_metrics)
                out["simplex_horizon_metrics"] = json.dumps(horizon_metrics)
                del result
            values = np.array([[r["cc"],r["nrmse"]] for r in horizon_metrics],dtype=float)
            if not np.isfinite(values).all():
                raise ValueError("Nonfinite metric in one or more frozen prediction horizons")
            out.update(Mean_CC=float(values[:,0].mean()), Mean_NRMSE=float(values[:,1].mean()), simplex_success=True)
        except Exception as exc:
            out["simplex_error"] = f"{type(exc).__name__}: {exc}"
        try:
            result = run_rqa(signal, m=M, tau=tau, l_min=L_MIN, v_min=V_MIN, target_rr=TARGET_RR)
            out.update(DET=result.det, Lmean=result.l_mean, LAM=result.lam, TT=result.tt,
                rqa_achieved_rr=result.achieved_rr, rqa_rr_exact=result.rr_exact,
                rqa_zero_distance_fraction=result.zero_distance_fraction, rqa_epsilon=result.epsilon)
            if not np.isfinite([result.det,result.l_mean,result.lam,result.tt]).all():
                raise ValueError("Nonfinite RQA metrics")
            out["rqa_success"] = True
            del result
        except Exception as exc:
            out["rqa_error"] = f"{type(exc).__name__}: {exc}"
        try:
            result = compute_rosenstein_lle(signal, sampling_rate=fs, m=M, tau_samples=tau,
                fit_start_s=FIT_START_S, fit_end_s=FIT_END_S, max_follow_s=MAX_FOLLOW_S, theiler_s=None,
                min_initial_pairs=MIN_INITIAL_PAIRS, min_fit_pairs=MIN_FIT_PAIRS, min_r2=MIN_R2)
            computed = bool(np.isfinite(result.lle) and np.isfinite(result.fit_r2))
            out.update(LLE=result.lle, LLE_fit_R2=result.fit_r2, LLE_fit_start_s=result.fit_start_s,
                LLE_fit_end_s=result.fit_end_s, LLE_pass_R2_090=bool(computed and result.fit_r2>=.90),
                LLE_pass_R2_095=bool(computed and result.fit_r2>=.95), lle_success=computed,
                LLE_theiler_samples=result.theiler_samples, LLE_theiler_s=result.theiler_s,
                LLE_n_pairs_initial=result.n_pairs_initial, LLE_n_pairs_fit_min=result.n_pairs_fit_min,
                LLE_core_valid=result.valid, LLE_qc_reason=result.qc_reason)
            if not computed:
                out["lle_error"] = result.qc_reason
            del result
        except Exception as exc:
            out["lle_error"] = f"{type(exc).__name__}: {exc}"
        out["warnings"] = " | ".join(str(w.message) for w in caught)
    out["error_message"] = " | ".join(f"{method}: {out[method+'_error']}" for method in ["simplex","rqa","lle"] if out[method+"_error"])
    out["elapsed_s"] = time.perf_counter()-started
    return out


def extraction_failure(message, fs):
    out = {k: np.nan for k in RESULT_FIELDS}
    out.update(simplex_success=False, rqa_success=False, lle_success=False,
        LLE_pass_R2_090=False, LLE_pass_R2_095=False, LLE_core_valid=False,
        signal_extraction_success=False, error_message=message,
        simplex_error=message, rqa_error=message, lle_error=message,
        simplex_horizons_completed=0, simplex_horizon_metrics="[]", warnings="", elapsed_s=0,
        m=M, tau_s=TAU_S, tau_samples=round(TAU_S*fs), requested_fit_start_s=FIT_START_S,
        requested_fit_end_s=FIT_END_S, max_follow_s=MAX_FOLLOW_S, target_rr=TARGET_RR)
    return out

def technical_summary(group, **identity):
    row = dict(identity, input_windows=len(group))
    for field in ["simplex_success","rqa_success","lle_success","LLE_pass_R2_090","LLE_pass_R2_095"]:
        row[field] = int(as_bool(group[field]).sum())
    for method in ["simplex","rqa","lle"]:
        failed = ~as_bool(group[f"{method}_success"])
        row[f"{method}_errors"] = int(failed.sum())
        row[f"{method}_error_sessions"] = ";".join(sorted(group.loc[failed,"session"].unique()))
    row["windows_missing_any_metric"] = int((~np.isfinite(group[METRICS].to_numpy(dtype=float))).any(axis=1).sum())
    row["identical_signal_reused"] = int(as_bool(group.reused_identical_signal).sum())
    return row

## Rule-specific execution

`ntsa_run(rule)` saves per-session CSVs, configuration/input fingerprints, execution counts, error details and validation checks inside the rule folder. The master execution table combines completed rules without overwriting other rules' summaries. Changing the order or contents of `label_rule` does not invalidate another rule's checkpoints.

Each completed window is persisted immediately. No unfinished rule is reported as complete. Six workers are used by default; set `N_JOBS` in configuration to change concurrency.

In [3]:
def check_rule_available(rule):
    """Check one rule without computing metrics or creating outputs."""
    if not isinstance(rule, str) or not re.fullmatch(r"[A-Za-z][A-Za-z0-9_]*", rule):
        raise ValueError("rule must be a simple folder name, e.g. 'T30' or 'S6'")
    root = INPUT_DIR / rule
    missing = [str(root / name) for name in ("filtered_sessions", "windows_60s") if not (root / name).is_dir()]
    if missing:
        raise FileNotFoundError(
            f"No Phase B0 dataset for {rule}: {', '.join(missing)}. "
            "Create that label-rule dataset first, or choose an existing rule. S6 is not an alias for S5."
        )
    for name in ("master_summary.csv", "per_session_summary.csv"):
        table = read_table(INPUT_DIR / name)
        if rule not in set(table.rule):
            raise ValueError(f"{rule} is absent from Phase B0 {name}; regenerate the B0 summary for that dataset.")
    return rule


def update_master_execution_summary():
    """Combine completed rule summaries; incomplete rules are not marked complete."""
    paths = sorted(OUTPUT_DIR.glob("*/execution_summary.csv"))
    if not paths:
        return pd.DataFrame()
    combined = pd.concat([read_table(path) for path in paths], ignore_index=True)
    assert combined.rule.is_unique
    combined.to_csv(OUTPUT_DIR / "master_execution_summary.csv", index=False)
    return combined

In [4]:
def ntsa_run(rule):
    """Run/resume one Phase B0 label rule and save its metrics and technical summaries."""
    RULES = (check_rule_available(rule),)
    b0 = read_table(INPUT_DIR / "master_summary.csv").set_index("rule")
    b0_sessions = read_table(INPUT_DIR / "per_session_summary.csv")
    inputs, coverage, file_hashes = [], [], {}
    for rule in RULES:
        paths = sorted((INPUT_DIR / rule / "windows_60s").glob("*.csv"), key=lambda p:int(p.stem.rsplit("_",1)[-1]))
        assert len(paths)==20
        counts = {"Awake":0, "Drowsy":0}
        for path in paths:
            table = read_table(path)
            selected = table.loc[as_bool(table.analysis_included)].copy()
            assert selected.window_id.is_unique
            assert (selected.rule==rule).all() and (selected.duration_s==60).all()
            session = path.stem.rsplit("_",1)[-1].zfill(2)
            ref = b0_sessions[(b0_sessions.rule==rule)&(b0_sessions.session==session)]
            assert len(ref)==1
            assert (selected.session==session).all()
            for state in counts:
                count = int(selected.state.eq(state).sum())
                assert count == int(ref[f"{state.lower()}_included_windows"].iloc[0])
                counts[state] += count
            inputs.append((rule, session, path))
            file_hashes[str(path.relative_to(INPUT_DIR))] = digest(path)
            # Include even sessions without selected windows in the input manifest.
            signal_path = INPUT_DIR / rule / "filtered_sessions" / path.name
            file_hashes[str(signal_path.relative_to(INPUT_DIR))] = digest(signal_path)
        expected = int(b0.loc[rule,"awake_included_windows"] + b0.loc[rule,"drowsy_included_windows"])
        assert sum(counts.values())==expected
        coverage.append(dict(rule=rule, input_windows=expected, awake_windows=counts["Awake"], drowsy_windows=counts["Drowsy"]))
    file_hashes["master_summary.csv"] = digest(INPUT_DIR/"master_summary.csv")
    file_hashes["per_session_summary.csv"] = digest(INPUT_DIR/"per_session_summary.csv")
    configuration = dict(m=M, tau_s=TAU_S, horizons_s=HORIZON_SECONDS.tolist(), simplex_theiler_s=SIMPLEX_THEILER_S,
        simplex_neighbors=M+1, simplex_normalization="window z-score ddof=0; NRMSE scale=1", simplex_validation="leave-one-out",
        distance="euclidean", target_rr=TARGET_RR, rqa_theiler="(m-1)*tau_samples", l_min=L_MIN, v_min=V_MIN,
        fit_start_s=FIT_START_S, fit_end_s=FIT_END_S, max_follow_s=MAX_FOLLOW_S, lle_theiler="spectral mean period per window",
        min_initial_pairs=MIN_INITIAL_PAIRS, min_fit_pairs=MIN_FIT_PAIRS, primary_r2=.90, sensitivity_r2=.95,
        core_hashes={str(Path(inspect.getfile(f)).relative_to(PHASE1_DIR)):digest(inspect.getfile(f)) for f in [simplex_projection,run_rqa,compute_rosenstein_lle]},
        primary_horizon_source=str(PRIMARY_SIMPLEX.relative_to(PHASE1_DIR)), primary_horizon_source_sha256=digest(PRIMARY_SIMPLEX),
        input_hashes=file_hashes, versions={"numpy":np.__version__,"pandas":pd.__version__,"scipy":scipy.__version__})
    CONFIG_HASH = hashlib.sha256(json.dumps(configuration,sort_keys=True).encode()).hexdigest()
    OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
    config_path = OUTPUT_DIR / rule / "run_configuration.json"
    config_path.parent.mkdir(parents=True, exist_ok=True)
    if config_path.exists():
        assert json.loads(config_path.read_text())==configuration, "Existing output belongs to different inputs/configuration"
    else:
        config_path.write_text(json.dumps(configuration,indent=2),encoding="utf-8")
    coverage = pd.DataFrame(coverage)
    coverage.to_csv(OUTPUT_DIR/rule/"input_coverage.csv",index=False)
    display(coverage)

    cache = {}
    # Populate the scalar cache from validated checkpoints; no P0 data is read.
    for rule, session, _ in inputs:
        path = OUTPUT_DIR/rule/f"session_{session}_ntsa.csv"
        if path.exists():
            old = read_table(path)
            assert set(RESULT_FIELDS).issubset(old.columns) and old.window_id.is_unique
            assert old.configuration_hash.eq(CONFIG_HASH).all()
            for row in old.to_dict("records"):
                if row["signal_extraction_success"] == True:
                    cache[row["computation_key"]] = {k:row[k] for k in RESULT_FIELDS}
    started = time.perf_counter()
    completed_count = 0
    with parallel_config(backend="loky", inner_max_num_threads=1):
        with Parallel(n_jobs=N_JOBS, return_as="generator_unordered", pre_dispatch=N_JOBS, batch_size=1) as pool:
            for rule, session, input_path in inputs:
                source = read_table(input_path)
                selected = source.loc[as_bool(source.analysis_included)].copy()
                outpath = OUTPUT_DIR/rule/f"session_{session}_ntsa.csv"
                outpath.parent.mkdir(parents=True,exist_ok=True)
                previous = read_table(outpath) if outpath.exists() else pd.DataFrame(columns=["window_id"])
                assert set(previous.window_id).issubset(set(selected.window_id))
                done = set(previous.window_id)
                signals = {}
                pending, consumers = {}, {}
                columns = list(source.columns) + RESULT_FIELDS
                assert len(columns)==len(set(columns)), "Metadata and result columns overlap"
                with outpath.open("a",newline="",encoding="utf-8") as stream:
                    writer = csv.DictWriter(stream,fieldnames=columns)
                    if stream.tell()==0:
                        writer.writeheader(); stream.flush()
                    def persist(meta,result,key="",signal_hash="",reused=False):
                        row = dict(meta)
                        row.update(result)
                        row.update(configuration_hash=CONFIG_HASH, computation_key=key, signal_sha256=signal_hash,
                            reused_identical_signal=reused,
                            computed_for_window_id=result.get("computed_for_window_id") if reused else meta["window_id"])
                        writer.writerow(row); stream.flush(); os.fsync(stream.fileno())
                    for meta in selected.to_dict("records"):
                        if meta["window_id"] in done:
                            continue
                        try:
                            signal_path = (INPUT_DIR/meta["signal_file"]).resolve()
                            assert signal_path.is_relative_to(INPUT_DIR.resolve())
                            if str(signal_path) not in signals:
                                signals[str(signal_path)] = pd.read_csv(signal_path,float_precision="round_trip")
                            frame = signals[str(signal_path)]
                            a,b = int(meta["window_start_index"]),int(meta["window_end_index"])
                            fs = float(meta["fs"])
                            assert np.isfinite(fs) and fs>0 and 0<=a<b<=len(frame)
                            window = frame.iloc[a:b]
                            assert b-a==int(meta["n_samples"])==round(60*fs)
                            assert window.Label.eq(meta["label"]).all() and window.segment_id.eq(meta["segment_id"]).all()
                            assert window.retained_interval_id.eq(meta["retained_interval_id"]).all()
                            assert np.array_equal(window.source_row_index,np.arange(meta["source_start_index"],meta["source_end_index"]))
                            assert window["Time (s)"].iloc[0]==meta["window_start_s"] and window["Time (s)"].iloc[-1]==meta["window_end_s"]
                            assert np.all(np.diff(window["Time (s)"])>0)
                            ref = b0_sessions[(b0_sessions.rule==rule)&(b0_sessions.session==session)].iloc[0]
                            assert fs==float(ref.fs)
                            signal = np.ascontiguousarray(window["PPG processed"].to_numpy(dtype=float))
                            assert np.isfinite(signal).all() and round(TAU_S*fs)>=1
                            signal_hash = hashlib.sha256(signal.tobytes()).hexdigest()
                            key = hashlib.sha256((CONFIG_HASH+float(fs).hex()+signal_hash).encode()).hexdigest()
                        except Exception as exc:
                            persist(meta,extraction_failure(f"input: {type(exc).__name__}: {exc}",float(meta["fs"])))
                            continue
                        if key in cache:
                            persist(meta,cache[key],key,signal_hash,True)
                        else:
                            consumers.setdefault(key,[]).append((meta,signal_hash))
                            pending.setdefault(key,(signal,fs))
                    def job(key,signal,fs):
                        return key,compute_metrics(signal,fs)
                    for key,result in pool(delayed(job)(key,*task) for key,task in pending.items()):
                        result["computed_for_window_id"] = consumers[key][0][0]["window_id"]
                        cache[key] = result
                        for index,(meta,signal_hash) in enumerate(consumers[key]):
                            persist(meta,result,key,signal_hash,index>0)
                actual = read_table(outpath)
                assert len(actual)==len(selected) and actual.window_id.is_unique
                completed_count += len(actual)
                print(f"{rule} session {session}: {len(actual)} rows saved; total {completed_count}/{coverage.input_windows.sum()} ({(time.perf_counter()-started)/60:.1f} min)",flush=True)
                del signals,pending,consumers
    print(f"All sensitivity windows saved. Unique scalar cache entries: {len(cache)}")

    all_results, per_session_rows = [], []
    for rule, session, source_path in inputs:
        source = read_table(source_path)
        expected = source.loc[as_bool(source.analysis_included)].sort_values("window_id").reset_index(drop=True)
        result = read_table(OUTPUT_DIR/rule/f"session_{session}_ntsa.csv").sort_values("window_id").reset_index(drop=True)
        assert len(result)==len(expected) and result.window_id.is_unique
        pd.testing.assert_frame_equal(result[list(source.columns)],expected,check_dtype=False)
        assert result.configuration_hash.eq(CONFIG_HASH).all()
        assert np.array_equal(result.tau_samples, np.rint(TAU_S*result.fs).astype(int))
        assert result.m.eq(M).all() and result.tau_s.eq(TAU_S).all()
        assert result.requested_fit_start_s.eq(FIT_START_S).all() and result.requested_fit_end_s.eq(FIT_END_S).all()
        for method,fields in [("simplex",["Mean_CC","Mean_NRMSE"]),("rqa",["DET","Lmean","LAM","TT"]),("lle",["LLE","LLE_fit_R2"])]:
            success=as_bool(result[f"{method}_success"])
            assert np.isfinite(result.loc[success,fields].to_numpy(float)).all()
        assert np.array_equal(as_bool(result.LLE_pass_R2_090), as_bool(result.lle_success)&result.LLE_fit_R2.ge(.90))
        assert np.array_equal(as_bool(result.LLE_pass_R2_095), as_bool(result.lle_success)&result.LLE_fit_R2.ge(.95))
        assert result.loc[as_bool(result.simplex_success),"simplex_horizons_completed"].eq(18).all()
        per_session_rows.append(technical_summary(result,rule=rule,session=session))
        all_results.append(result)
    all_results = pd.concat(all_results,ignore_index=True)
    assert all_results.window_id.is_unique
    for relative,expected_hash in file_hashes.items():
        assert digest(INPUT_DIR/relative)==expected_hash, f"Input changed: {relative}"
    master_rows=[]
    for rule in RULES:
        group=all_results[all_results.rule==rule]
        assert len(group)==int(coverage.loc[coverage.rule==rule,"input_windows"].iloc[0])
        summary=technical_summary(group,rule=rule,n_sessions=group.session.nunique())
        master_rows.append(summary)
        pd.DataFrame([summary]).to_csv(OUTPUT_DIR/rule/"execution_summary.csv",index=False)
        pd.DataFrame([r for r in per_session_rows if r["rule"]==rule]).to_csv(OUTPUT_DIR/rule/"session_execution_summary.csv",index=False)
        failed=group.loc[~(as_bool(group.simplex_success)&as_bool(group.rqa_success)&as_bool(group.lle_success)),["session","window_id","simplex_error","rqa_error","lle_error","error_message"]]
        failed.to_csv(OUTPUT_DIR/rule/"method_errors.csv",index=False)
    master_execution_summary=pd.DataFrame(master_rows)
    update_master_execution_summary()
    validation = pd.DataFrame([{"check":check,"passed":True} for check in ["B0 input count and identities", "Original metadata preserved", "No duplicate window_id", "Correct fs and tau conversion", "Fixed parameters", "Finite metrics on success", "LLE QC flags", "Input file hashes unchanged"]])
    validation.to_csv(OUTPUT_DIR/rule/"validation_summary.csv",index=False)
    display(master_execution_summary)
    display(validation)
    print(f"{rule} complete: window-level metrics and technical counts only.")
    return master_execution_summary


## Chạy các label rule

Chỉnh danh sách bên dưới rồi chạy cell cuối. Tên rule là chuỗi Python, ví dụ `"T30"`. `ntsa_run(rule)` tự kiểm tra dữ liệu, chạy hoặc tiếp tục checkpoint, lưu kết quả và các bảng execution summary của rule đó.

S6 cần có dataset Phase B0 riêng tại `results/data_label_sensitive/S6` và được ghi nhận trong các bảng summary đầu vào. Hiện có T30/T60/S3/S5; nếu muốn chạy S5, thay `"S6"` bằng `"S5"`. Không tự đổi rule và không tạo lại dataset tại bước này.

In [5]:
label_rule = ["T30", "T60", "S3", "S5"]
for rule in label_rule:
    ntsa_run(rule)

,rule,input_windows,awake_windows,drowsy_windows
0,T30,860,586,274


T30 session 01: 27 rows saved; total 27/860 (0.3 min)
T30 session 04: 42 rows saved; total 69/860 (0.5 min)
T30 session 05: 49 rows saved; total 118/860 (0.7 min)
T30 session 06: 47 rows saved; total 165/860 (0.9 min)
T30 session 07: 40 rows saved; total 205/860 (1.1 min)
T30 session 08: 44 rows saved; total 249/860 (1.3 min)
T30 session 09: 63 rows saved; total 312/860 (1.6 min)
T30 session 10: 35 rows saved; total 347/860 (1.8 min)
T30 session 11: 37 rows saved; total 384/860 (2.0 min)
T30 session 12: 48 rows saved; total 432/860 (2.2 min)
T30 session 13: 45 rows saved; total 477/860 (2.4 min)
T30 session 14: 45 rows saved; total 522/860 (2.6 min)
T30 session 15: 34 rows saved; total 556/860 (2.8 min)
T30 session 17: 39 rows saved; total 595/860 (3.0 min)
T30 session 18: 41 rows saved; total 636/860 (3.2 min)
T30 session 19: 43 rows saved; total 679/860 (3.4 min)
T30 session 21: 38 rows saved; total 717/860 (3.5 min)
T30 session 22: 42 rows saved; total 759/860 (3.7 min)
T30 session 

,rule,n_sessions,input_windows,simplex_success,rqa_success,lle_success,LLE_pass_R2_090,LLE_pass_R2_095,simplex_errors,simplex_error_sessions,rqa_errors,rqa_error_sessions,lle_errors,lle_error_sessions,windows_missing_any_metric,identical_signal_reused
0,T30,20,860,860,860,860,832,692,0,,0,,0,,0,0


,check,passed
0,B0 input count and identities,True
1,Original metadata preserved,True
2,No duplicate window_id,True
3,Correct fs and tau conversion,True
4,Fixed parameters,True
5,Finite metrics on success,True
6,LLE QC flags,True
7,Input file hashes unchanged,True


T30 complete: window-level metrics and technical counts only.


...

,rule,input_windows,awake_windows,drowsy_windows
0,S5,846,581,265


S5 session 01: 19 rows saved; total 19/846 (0.2 min)
S5 session 04: 40 rows saved; total 59/846 (0.4 min)
S5 session 05: 50 rows saved; total 109/846 (0.6 min)
S5 session 06: 47 rows saved; total 156/846 (0.9 min)
S5 session 07: 43 rows saved; total 199/846 (1.1 min)
S5 session 08: 41 rows saved; total 240/846 (1.3 min)
S5 session 09: 63 rows saved; total 303/846 (1.6 min)
S5 session 10: 32 rows saved; total 335/846 (1.8 min)
S5 session 11: 39 rows saved; total 374/846 (2.0 min)
S5 session 12: 52 rows saved; total 426/846 (2.2 min)
S5 session 13: 49 rows saved; total 475/846 (2.5 min)
S5 session 14: 43 rows saved; total 518/846 (2.7 min)
S5 session 15: 27 rows saved; total 545/846 (2.8 min)
S5 session 17: 38 rows saved; total 583/846 (3.0 min)
S5 session 18: 42 rows saved; total 625/846 (3.2 min)
S5 session 19: 45 rows saved; total 670/846 (3.4 min)
S5 session 21: 34 rows saved; total 704/846 (3.6 min)
S5 session 22: 39 rows saved; total 743/846 (3.8 min)
S5 session 23: 66 rows saved; 

,rule,n_sessions,input_windows,simplex_success,rqa_success,lle_success,LLE_pass_R2_090,LLE_pass_R2_095,simplex_errors,simplex_error_sessions,rqa_errors,rqa_error_sessions,lle_errors,lle_error_sessions,windows_missing_any_metric,identical_signal_reused
0,S5,20,846,846,846,846,819,689,0,,0,,0,,0,0


,check,passed
0,B0 input count and identities,True
1,Original metadata preserved,True
2,No duplicate window_id,True
3,Correct fs and tau conversion,True
4,Fixed parameters,True
5,Finite metrics on success,True
6,LLE QC flags,True
7,Input file hashes unchanged,True


S5 complete: window-level metrics and technical counts only.
